# GroupBy and Aggregation

`groupby` answers the most common business and ML question: **"for each X, what is Y?"**

- For each **region**, what is the total revenue?
- For each **customer**, how many orders and what average amount?
- For each **category**, what fraction of orders is returned?

It is also the main tool for **feature engineering**: turning thousands of transaction rows into **one row per customer** with features a model can learn from.

**Contents**
1. The idea: split, apply, combine
2. Our datasets
3. Basic `groupby`
4. Multiple keys, `as_index`, text columns
5. `agg`: lists, dicts and **named aggregation**
6. `size` vs `count` vs `nunique`
7. `transform`: group statistics on every row
8. `filter`, `apply`, `head`, `nth`, `cumcount`
9. Grouping by dates, bins, conditions
10. Categorical groups and `observed`
11. Preview: `pivot_table` and `unstack`
12. Speed: built-in vs `lambda`
13. Mini AI/ML example: customer features and target encoding
14. Pandas 3.x notes
15. Common mistakes
16. Interview questions
17. Practice questions with solutions
18. Cheat sheet

In [2]:
import time
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. The idea: split, apply, combine

```
 original table            SPLIT into groups       APPLY a function        COMBINE the results
 region  revenue           North: 100, 800, ...    sum -> 2270             region  revenue
 North   100               South: 500, 2400, ...   sum -> 2980             North   2270
 South   500       --->    East : 250, 500, ...    sum -> 1550     --->    South   2980
 North   800               ...                                             East    1550
 East    250
 ...
```

1. **Split** the rows into groups by one or more keys
2. **Apply** a function to each group (sum, mean, count, your own function)
3. **Combine** the results into a new table

SQL people: this is exactly `GROUP BY`.

---
# 2. Our datasets

A small table to learn the syntax, and a bigger transaction table for the ML part (created later).

In [3]:
sales = pd.DataFrame({
    "region":  ["North", "South", "North", "East", "South", "North", "East", "South", "East", "North"],
    "product": ["Pen", "Book", "Bag", "Pen", "Bag", "Book", "Book", "Pen", "Bag", "Pen"],
    "units":   [10, 2, 1, 25, 3, 5, 2, 8, 1, 12],
    "revenue": [100, 500, 800, 250, 2400, 1250, 500, 80, 800, 120],
    "rating":  [4.5, 4.0, np.nan, 3.5, 4.2, 4.8, np.nan, 3.9, 4.1, 4.4],
})
sales

,region,product,units,revenue,rating
0,North,Pen,10,100,4.5
1,South,Book,2,500,4.0
2,North,Bag,1,800,NaN
3,East,Pen,25,250,3.5
4,South,Bag,3,2400,4.2
5,North,Book,5,1250,4.8
6,East,Book,2,500,NaN
7,South,Pen,8,80,3.9
8,East,Bag,1,800,4.1
9,North,Pen,12,120,4.4


---
# 3. Basic `groupby`

## 3.1 The manual way, to see what groupby does

In [4]:
# Without groupby: loop over the regions yourself
for region in sales["region"].unique():
    part = sales[sales["region"] == region]
    print(region, part["revenue"].sum())

North 2270
South 2980
East 1550


In [5]:
# With groupby: one line
sales.groupby("region")["revenue"].sum()

region
East     1550
North    2270
South    2980
Name: revenue, dtype: int64

Read it as a sentence: **"group by region, take the revenue column, sum it."**

## 3.2 What is a `groupby` object?

It is a **lazy** object: nothing is calculated until you apply a function.

In [6]:
g = sales.groupby("region")

print(type(g))
print("number of groups:", g.ngroups)
print("group sizes:", g.size().to_dict())

<class 'pandas.api.typing.DataFrameGroupBy'>
number of groups: 3
group sizes: {'East': 3, 'North': 4, 'South': 3}


In [7]:
# One group as a normal DataFrame
g.get_group("East")

,region,product,units,revenue,rating
3,East,Pen,25,250,3.5
6,East,Book,2,500,NaN
8,East,Bag,1,800,4.1


In [8]:
# Iterating gives (name, sub-table) pairs
for name, part in g:
    print(name, "->", len(part), "rows, revenue", part["revenue"].sum())

East -> 3 rows, revenue 1550
North -> 4 rows, revenue 2270
South -> 3 rows, revenue 2980


## 3.3 Common aggregation functions

| Function | Meaning |
|---|---|
| `sum`, `mean`, `median` | total, average, middle value |
| `min`, `max` | smallest, largest |
| `std`, `var` | spread |
| `count` | non-missing values |
| `size` | number of rows (including missing) |
| `nunique` | number of different values |
| `first`, `last` | first / last row of each group |
| `prod`, `sem`, `skew` | product, standard error, skewness |

In [9]:
rev = sales.groupby("region")["revenue"]

print(rev.sum().to_dict())
print(rev.mean().round(1).to_dict())
print(rev.max().to_dict())
print(sales.groupby("region")["product"].nunique().to_dict())

{'East': 1550, 'North': 2270, 'South': 2980}
{'East': 516.7, 'North': 567.5, 'South': 993.3}
{'East': 800, 'North': 1250, 'South': 2400}
{'East': 3, 'North': 3, 'South': 3}


In [10]:
# Several columns at once: pass a list
sales.groupby("region")[["units", "revenue"]].sum()

,units,revenue
region,,
East,28,1550
North,28,2270
South,13,2980


---
# 4. Multiple keys, `as_index`, text columns

## 4.1 Group by more than one column

In [11]:
sales.groupby(["region", "product"])["revenue"].sum()

region  product
East    Bag         800
        Book        500
        Pen         250
North   Bag         800
        Book       1250
        Pen         220
South   Bag        2400
        Book        500
        Pen          80
Name: revenue, dtype: int64

The result has a **MultiIndex** (two index levels). Two common ways to make it a normal table:

In [12]:
# Option 1: reset_index
sales.groupby(["region", "product"])["revenue"].sum().reset_index()

,region,product,revenue
0,East,Bag,800
1,East,Book,500
2,East,Pen,250
3,North,Bag,800
4,North,Book,1250
5,North,Pen,220
6,South,Bag,2400
7,South,Book,500
8,South,Pen,80


In [13]:
# Option 2: as_index=False from the start
sales.groupby(["region", "product"], as_index=False)["revenue"].sum()

,region,product,revenue
0,East,Bag,800
1,East,Book,500
2,East,Pen,250
3,North,Bag,800
4,North,Book,1250
5,North,Pen,220
6,South,Bag,2400
7,South,Book,500
8,South,Pen,80


## 4.2 Useful options

- `as_index=False`: keep the group keys as normal columns
- `sort=False`: do not sort the groups (keeps order of first appearance, slightly faster)
- `dropna=False`: keep rows whose key is NaN as their own group (by default they are **dropped silently**)

In [14]:
print(sales.groupby("region", sort=False)["revenue"].sum().index.tolist())

['North', 'South', 'East']


In [15]:
tmp = sales.copy()
tmp.loc[0, "region"] = np.nan

print(tmp.groupby("region")["revenue"].sum())                    # the NaN row disappears
print()
print(tmp.groupby("region", dropna=False)["revenue"].sum())      # it stays as a NaN group

region
East     1550
North    2170
South    2980
Name: revenue, dtype: int64

region
East     1550
North    2170
South    2980
NaN       100
Name: revenue, dtype: int64


## 4.3 Text columns and numeric functions

`groupby(...).mean()` on the **whole** DataFrame tries to average every column, including text ones. In current pandas this raises an error. Select the columns you need, or use `numeric_only=True`.

In [16]:
try:
    sales.groupby("region").mean()
except Exception as e:
    print("Error:", type(e).__name__)

Error: TypeError


In [17]:
sales.groupby("region").mean(numeric_only=True).round(2)

,units,revenue,rating
region,,,
East,9.33,516.67,3.80
North,7.00,567.50,4.57
South,4.33,993.33,4.03


---
# 5. `agg`: lists, dicts and named aggregation

## 5.1 Several functions on one column

In [18]:
sales.groupby("region")["revenue"].agg(["count", "sum", "mean", "max"])

,count,sum,mean,max
region,,,,
East,3,1550,516.666667,800
North,4,2270,567.500000,1250
South,3,2980,993.333333,2400


## 5.2 Different functions for different columns (dict)

In [19]:
sales.groupby("region").agg({"units": "sum", "revenue": ["sum", "mean"], "rating": "mean"})

units revenue                rating
         sum     sum        mean      mean
region                                    
East      28    1550  516.666667  3.800000
North     28    2270  567.500000  4.566667
South     13    2980  993.333333  4.033333

The columns are now a **MultiIndex** (`revenue` -> `sum`, `mean`), which is awkward to use. The fix is **named aggregation**.

## 5.3 Named aggregation (the recommended way)

Syntax: `new_name=("column", "function")`. You choose clear column names, and the result is a flat table.

In [20]:
summary = sales.groupby("region").agg(
    n_orders=("revenue", "count"),
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum"),
    avg_revenue=("revenue", "mean"),
    avg_rating=("rating", "mean"),
    n_products=("product", "nunique"),
)
summary.round(2)

,n_orders,total_units,total_revenue,avg_revenue,avg_rating,n_products
region,,,,,,
East,3,28,1550,516.67,3.80,3
North,4,28,2270,567.50,4.57,3
South,3,13,2980,993.33,4.03,3


In [21]:
# With your own functions
def revenue_range(s):
    return s.max() - s.min()

sales.groupby("region").agg(
    top_order=("revenue", "max"),
    spread=("revenue", revenue_range),
    p90=("revenue", lambda s: s.quantile(0.9)),
    best_product=("product", lambda s: s.mode().iloc[0]),     # most frequent product
).round(1)

,top_order,spread,p90,best_product
region,,,,
East,800,550,740.0,Bag
North,1250,1150,1115.0,Pen
South,2400,2320,2020.0,Bag


If the old dict style gave you a MultiIndex, you can flatten it like this:

In [22]:
multi = sales.groupby("region").agg({"revenue": ["sum", "mean"], "units": "sum"})
multi.columns = ["_".join(col) for col in multi.columns]
multi

,revenue_sum,revenue_mean,units_sum
region,,,
East,1550,516.666667,28
North,2270,567.500000,28
South,2980,993.333333,13


---
# 6. `size` vs `count` vs `nunique`

| | Counts | NaN |
|---|---|---|
| `size()` | **rows** in the group | included |
| `count()` | **non-missing values** of a column | not counted |
| `nunique()` | **distinct** values | not counted (by default) |

The `rating` column has missing values, so the difference is visible:

In [23]:
pd.DataFrame({
    "size":     sales.groupby("region").size(),
    "count":    sales.groupby("region")["rating"].count(),
    "nunique":  sales.groupby("region")["rating"].nunique(),
})

,size,count,nunique
region,,,
East,3,2,2
North,4,3,3
South,3,3,3


In [24]:
# size() as a normal column
sales.groupby("region").size().reset_index(name="n_orders")

,region,n_orders
0,East,3
1,North,4
2,South,3


In [25]:
# Same idea as value_counts
print(sales["region"].value_counts().to_dict())
print(sales.groupby("region").size().to_dict())

{'North': 4, 'South': 3, 'East': 3}
{'East': 3, 'North': 4, 'South': 3}


---
# 7. `transform`: group statistics on every row

`agg` **shrinks** the table (one row per group). `transform` **keeps every row** and fills in the group result. That makes it perfect for features.

In [26]:
g = sales.groupby("region")["revenue"]

print("agg      :", len(g.agg("sum")), "rows")
print("transform:", len(g.transform("sum")), "rows (same as the original table)")

agg      : 3 rows
transform: 10 rows (same as the original table)


In [27]:
d = sales.copy()

d["region_total"] = d.groupby("region")["revenue"].transform("sum")
d["share_in_region_%"] = (d["revenue"] / d["region_total"] * 100).round(1)       # percent of group total
d["vs_region_avg"] = (d["revenue"] - d.groupby("region")["revenue"].transform("mean")).round(1)
d["rank_in_region"] = d.groupby("region")["revenue"].rank(ascending=False, method="first").astype(int)

d[["region", "product", "revenue", "region_total", "share_in_region_%", "vs_region_avg", "rank_in_region"]]

,region,product,revenue,region_total,share_in_region_%,vs_region_avg,rank_in_region
0,North,Pen,100,2270,4.4,-467.5,4
1,South,Book,500,2980,16.8,-493.3,2
2,North,Bag,800,2270,35.2,232.5,2
3,East,Pen,250,1550,16.1,-266.7,3
4,South,Bag,2400,2980,80.5,1406.7,1
5,North,Book,1250,2270,55.1,682.5,1
6,East,Book,500,1550,32.3,-16.7,2
7,South,Pen,80,2980,2.7,-913.3,3
8,East,Bag,800,1550,51.6,283.3,1
9,North,Pen,120,2270,5.3,-447.5,3


In [28]:
# Fill missing ratings with the average rating of the same region
d["rating_filled"] = d["rating"].fillna(d.groupby("region")["rating"].transform("mean")).round(2)
d[["region", "rating", "rating_filled"]]

,region,rating,rating_filled
0,North,4.5,4.50
1,South,4.0,4.00
2,North,NaN,4.57
3,East,3.5,3.50
4,South,4.2,4.20
5,North,4.8,4.80
6,East,NaN,3.80
7,South,3.9,3.90
8,East,4.1,4.10
9,North,4.4,4.40


In [29]:
# z-score inside each group
d["revenue_z"] = d.groupby("region")["revenue"].transform(lambda s: (s - s.mean()) / s.std()).round(2)
d[["region", "revenue", "revenue_z"]].head(5)

,region,revenue,revenue_z
0,North,100,-0.84
1,South,500,-0.40
2,North,800,0.42
3,East,250,-0.97
4,South,2400,1.14


---
# 8. `filter`, `apply`, `head`, `nth`, `cumcount`

## 8.1 `filter`: keep or drop **whole groups**

The function receives each group and must return True (keep) or False (drop).

In [30]:
# Keep only regions with at least 3 orders
sales.groupby("region").filter(lambda grp: len(grp) >= 3)

,region,product,units,revenue,rating
0,North,Pen,10,100,4.5
1,South,Book,2,500,4.0
2,North,Bag,1,800,NaN
3,East,Pen,25,250,3.5
4,South,Bag,3,2400,4.2
5,North,Book,5,1250,4.8
6,East,Book,2,500,NaN
7,South,Pen,8,80,3.9
8,East,Bag,1,800,4.1
9,North,Pen,12,120,4.4


In [31]:
# Keep regions whose total revenue is above 2500
sales.groupby("region").filter(lambda grp: grp["revenue"].sum() > 2500)[["region", "revenue"]]

,region,revenue
1,South,500
4,South,2400
7,South,80


## 8.2 `apply`: any function on each group

Slow and flexible. Use it **only** when `agg` / `transform` / `filter` cannot express the logic.

In [32]:
# The best order of each region (a whole row per group)
sales.groupby("region").apply(lambda grp: grp.nlargest(1, "revenue"))

,,product,units,revenue,rating
region,,,,,
East,8,Bag,1,800,4.1
North,5,Book,5,1250,4.8
South,4,Bag,3,2400,4.2


In [33]:
# A custom summary per group
def describe_group(grp):
    return pd.Series({
        "orders": len(grp),
        "revenue_per_unit": round(grp["revenue"].sum() / grp["units"].sum(), 1),
    })

sales.groupby("region").apply(describe_group)

,orders,revenue_per_unit
region,,
East,3.0,55.4
North,4.0,81.1
South,3.0,229.2


## 8.3 `head`, `tail`, `nth`, `cumcount`, `ngroup`

In [34]:
# First 2 rows of each group (in the table order)
sales.groupby("region").head(2)[["region", "product", "revenue"]]

,region,product,revenue
0,North,Pen,100
1,South,Book,500
2,North,Bag,800
3,East,Pen,250
4,South,Bag,2400
6,East,Book,500


In [35]:
# The n-th row of each group (0 = first)
sales.groupby("region").nth(0)[["region", "product", "revenue"]]

,region,product,revenue
0,North,Pen,100
1,South,Book,500
3,East,Pen,250


In [36]:
d = sales[["region", "product"]].copy()
d["order_no_in_region"] = d.groupby("region").cumcount() + 1       # 1, 2, 3... inside each group
d["group_id"] = d.groupby("region").ngroup()                       # a number per group
d

,region,product,order_no_in_region,group_id
0,North,Pen,1,1
1,South,Book,1,2
2,North,Bag,2,1
3,East,Pen,1,0
4,South,Bag,2,2
5,North,Book,3,1
6,East,Book,2,0
7,South,Pen,3,2
8,East,Bag,3,0
9,North,Pen,4,1


---
# 9. Grouping by dates, bins and conditions

The key does not have to be a column. It can be any Series, function or computed grouping.

## 9.1 By date parts and by time periods

In [37]:
rng = np.random.default_rng(2)
daily = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=90),
    "revenue": rng.integers(80, 200, 90),
})

# by a date part
print(daily.groupby(daily["date"].dt.month_name())["revenue"].sum())

date
February    4201
January     4260
March       4479
Name: revenue, dtype: int64


In [38]:
# by a time period with Grouper (ME = month end)
daily.groupby(pd.Grouper(key="date", freq="ME"))["revenue"].agg(["sum", "mean"]).round(1)

,sum,mean
date,,
2026-01-31,4260,137.4
2026-02-28,4201,150.0
2026-03-31,4479,144.5


In [39]:
# resample does the same on a date index; other frequencies: "W" week, "D" day, "QE" quarter end
daily.set_index("date")["revenue"].resample("W").sum().head(4)

date
2026-01-04    499
2026-01-11    980
2026-01-18    997
2026-01-25    903
Freq: W-SUN, Name: revenue, dtype: int64

## 9.2 By bins and by a condition

In [40]:
# By value ranges (bins made with pd.cut)
size_band = pd.cut(sales["units"], bins=[0, 2, 9, 100], labels=["tiny", "small", "bulk"])
sales.groupby(size_band, observed=True)["revenue"].agg(["count", "sum"])

,count,sum
units,,
tiny,4,2600
small,3,3730
bulk,3,470


In [41]:
# By a True/False condition
sales.groupby(sales["revenue"] > 500)["units"].sum()

revenue
False    59
True     10
Name: units, dtype: int64

In [42]:
# By a mapping: translate the key on the fly
state = {"North": "UP", "South": "TN", "East": "WB"}
sales.groupby(sales["region"].map(state))["revenue"].sum()

region
TN    2980
UP    2270
WB    1550
Name: revenue, dtype: int64

---
# 10. Categorical groups and `observed`

If the key is a **categorical** column that has categories without any rows, pandas can show those empty categories too. In **pandas 3 the default is `observed=True`** (only categories that really occur).

In [43]:
levels = pd.DataFrame({
    "grade": pd.Categorical(["A", "B", "A"], categories=["A", "B", "C"], ordered=True),
    "score": [90, 75, 95],
})

print(levels.groupby("grade")["score"].mean())                      # C is not shown
print()
print(levels.groupby("grade", observed=False)["score"].mean())      # C is shown with NaN

grade
A    92.5
B    75.0
Name: score, dtype: float64

grade
A    92.5
B    75.0
C     NaN
Name: score, dtype: float64


Use `observed=False` when you **want** the empty categories (for example a report that must always list every grade).

# 11. Preview: `pivot_table` and `unstack`

A groupby result with two keys can be turned into a **grid** (full chapter 12):

In [44]:
sales.groupby(["region", "product"])["revenue"].sum().unstack(fill_value=0)

product,Bag,Book,Pen
region,,,
East,800,500,250
North,800,1250,220
South,2400,500,80


In [45]:
sales.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum", fill_value=0, margins=True)

product,Bag,Book,Pen,All
region,,,,
East,800,500,250,1550
North,800,1250,220,2270
South,2400,500,80,2980
All,4000,2250,550,6800


---
# 12. Speed: built-in vs `lambda`

Built-in aggregations (`"mean"`, `"sum"`) run in fast compiled code. A Python `lambda` is called **once per group**.

In [46]:
rng = np.random.default_rng(0)
n = 300_000
big = pd.DataFrame({
    "key": rng.integers(0, 3000, n),
    "value": rng.random(n),
})

times = {}

start = time.perf_counter()
r1 = big.groupby("key")["value"].mean()
times['built-in mean()'] = time.perf_counter() - start

start = time.perf_counter()
r2 = big.groupby("key")["value"].agg(lambda s: s.mean())
times["agg(lambda)"] = time.perf_counter() - start

start = time.perf_counter()
r3 = big.groupby("key")["value"].apply(lambda s: s.mean())
times["apply(lambda)"] = time.perf_counter() - start

res = pd.DataFrame({"seconds": times})
res["times_slower"] = (res["seconds"] / res["seconds"].min()).round(0)
print(np.allclose(r1, r2), np.allclose(r1, r3))
res

True True


,seconds,times_slower
built-in mean(),0.004502,1.0
agg(lambda),0.041937,9.0
apply(lambda),0.041932,9.0


Rule: **use the built-in name when it exists** (`"mean"`, `"sum"`, `"max"`, `"nunique"`, `"first"`...). Use a lambda only for logic that has no built-in version (like the mode, or a custom percentile).

---
# 13. Mini AI/ML example: customer features and target encoding

ML models need **one row per entity** (a customer, a product, a device). Raw data has **many rows per entity** (transactions). `groupby` is the bridge.

## 13.1 Transaction data (400 orders from 40 customers, some very active, some rare)

In [47]:
rng = np.random.default_rng(11)
n = 400

activity = rng.dirichlet(np.ones(40) * 1.5)       # some customers order a lot, some rarely (like real life)

orders = pd.DataFrame({
    "order_id": np.arange(n),
    "customer_id": rng.choice([f"C{i:02d}" for i in range(1, 41)], n, p=activity),
    "date": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "category": rng.choice(["Electronics", "Grocery", "Fashion", "Home"], n, p=[0.2, 0.4, 0.25, 0.15]),
    "amount": rng.lognormal(6, 0.8, n).round(0),
    "returned": rng.random(n) < 0.12,
}).sort_values("date").reset_index(drop=True)

orders.head()

,order_id,customer_id,date,category,amount,returned
0,338,C06,2026-01-01,Grocery,1166.0,False
1,15,C33,2026-01-01,Electronics,4127.0,False
2,329,C16,2026-01-02,Grocery,113.0,False
3,217,C17,2026-01-02,Grocery,562.0,False
4,13,C27,2026-01-03,Electronics,1108.0,False


## 13.2 RFM features (Recency, Frequency, Monetary)

The classic customer features. Problem: **predict which customers will buy again in the next 14 days**.

**Time rule (no leakage):** features use only orders **before** the cutoff date. The label uses orders **after** it.

In [48]:
cutoff = pd.Timestamp("2026-05-01")

history = orders[orders["date"] < cutoff]                                            # what we know at prediction time
future = orders[(orders["date"] >= cutoff) & (orders["date"] < cutoff + pd.Timedelta(days=14))]   # the next 14 days

features = history.groupby("customer_id").agg(
    frequency=("order_id", "count"),
    monetary=("amount", "sum"),
    avg_order=("amount", "mean"),
    max_order=("amount", "max"),
    last_order=("date", "max"),
    first_order=("date", "min"),
    n_categories=("category", "nunique"),
    return_rate=("returned", "mean"),
)

features["recency_days"] = (cutoff - features["last_order"]).dt.days          # days since last purchase
features["tenure_days"] = (cutoff - features["first_order"]).dt.days
features = features.drop(columns=["last_order", "first_order"])

# label: did the customer buy again within 14 days after the cutoff?
features["bought_again"] = features.index.isin(future["customer_id"]).astype(int)

print(features.shape)
features.round(2).head()

(37, 9)


,frequency,monetary,avg_order,max_order,n_categories,return_rate,recency_days,tenure_days,bought_again
customer_id,,,,,,,,,
C01,4,1487.0,371.75,703.0,2,0.00,41,84,1
C02,18,10926.0,607.00,2654.0,4,0.11,9,109,1
C03,4,1130.0,282.50,365.0,2,0.25,14,101,0
C04,6,3828.0,638.00,1902.0,3,0.17,16,95,1
C05,7,3242.0,463.14,1240.0,3,0.14,5,109,1


In [49]:
# Do the features separate the two groups? (groupby on the label)
features.groupby("bought_again").agg(
    customers=("frequency", "size"),
    avg_frequency=("frequency", "mean"),
    avg_recency_days=("recency_days", "mean"),
    avg_monetary=("monetary", "mean"),
).round(1)

,customers,avg_frequency,avg_recency_days,avg_monetary
bought_again,,,,
0,16,5.3,33.9,2957.8
1,21,8.6,17.8,4920.1


Now `features` is a proper ML table: **one row per customer**, numeric features, one label. Hundreds of transactions became one row per customer. Customers who buy again usually have a higher `frequency` and a lower `recency_days` (they were active recently): the features carry real signal.

Category mix per customer (a group + pivot idea) is another common feature block:

In [50]:
mix = (history.groupby(["customer_id", "category"])["amount"].sum()
              .unstack(fill_value=0)
              .add_prefix("spend_"))
mix.head(3)

category,spend_Electronics,spend_Fashion,spend_Grocery,spend_Home
customer_id,,,,
C01,0.0,784.0,703.0,0.0
C02,2911.0,4039.0,3172.0,804.0
C03,327.0,0.0,803.0,0.0


In [51]:
model_table = features.join(mix)
print(model_table.shape)
model_table.columns.tolist()

(37, 13)


['frequency',
 'monetary',
 'avg_order',
 'max_order',
 'n_categories',
 'return_rate',
 'recency_days',
 'tenure_days',
 'bought_again',
 'spend_Electronics',
 'spend_Fashion',
 'spend_Grocery',
 'spend_Home']

## 13.3 Target encoding, and why it leaks

**Target encoding** replaces a category by the **average of the target** in that category (for example "return rate of this customer"). It is powerful, and also a classic source of **leakage**.

Task here: predict whether an order is `returned`. In our synthetic data, returns are **pure chance** (12% for everyone), so **no real pattern exists**. A good method must find **no relationship**.

In [52]:
# WRONG: the mean includes the row's OWN label (computed on all rows, then used on the same rows)
orders["cust_return_rate_LEAKY"] = orders.groupby("customer_id")["returned"].transform("mean")

leaky_corr = orders["cust_return_rate_LEAKY"].corr(orders["returned"].astype(float))
print("leaky feature correlation with the target:", round(leaky_corr, 3))

leaky feature correlation with the target: 0.313


In [53]:
# RIGHT: learn the mean from the TRAIN period only, then apply it to the TEST period
train = orders[orders["date"] < cutoff]
test = orders[orders["date"] >= cutoff].copy()

rate_from_train = train.groupby("customer_id")["returned"].mean()
global_rate = train["returned"].mean()

test["cust_return_rate"] = test["customer_id"].map(rate_from_train).fillna(global_rate)      # new customers -> global rate

honest_corr = test["cust_return_rate"].corr(test["returned"].astype(float))
print("honest feature correlation on test data:", round(honest_corr, 3))

# How big can a correlation be by pure chance with this few rows? Shuffle the labels many times.
rng2 = np.random.default_rng(0)
labels = test["returned"].to_numpy().astype(float)
chance = [test["cust_return_rate"].reset_index(drop=True).corr(pd.Series(rng2.permutation(labels))) for _ in range(500)]
print("95% of pure-chance correlations are within +/-", round(np.percentile(np.abs(chance), 95), 3))

honest feature correlation on test data: 0.124
95% of pure-chance correlations are within +/- 0.152


The leaky version looks like a useful feature (clearly positive correlation), but it is only **memorizing the answer**: each order's own label is inside its own feature. The honest version, measured on data it has never seen, stays **inside the range of pure chance** (see the shuffle test above), which is exactly right, because we built the data without any pattern.

**Rules for target encoding:**
1. Learn the group mean from the **train** data only (better: **out-of-fold** inside the train set).
2. Apply with `map` to the test/new data. Unseen categories get the **global mean**.
3. Use **smoothing** for small groups, so a customer with 1 order does not get a 0% or 100% rate:

`smoothed = (n * group_mean + m * global_mean) / (n + m)` where `n` is the group size and `m` is the smoothing strength (like 10).

In [54]:
stats = train.groupby("customer_id")["returned"].agg(n="size", mean="mean")
m = 10
stats["smoothed_rate"] = (stats["n"] * stats["mean"] + m * global_rate) / (stats["n"] + m)

stats.sort_values("n").head(5).round(3)

,n,mean,smoothed_rate
customer_id,,,
C32,1,0.0,0.12
C24,1,0.0,0.12
C15,1,0.0,0.12
C11,1,0.0,0.12
C14,2,0.0,0.11


The customers with few orders are pulled strongly toward the global rate, the ones with many orders keep their own rate.

---
# 14. Pandas 3.x notes

- **`groupby(...).apply(func)`** passes each group **without** the grouping column(s) (the old `include_groups=True` is no longer allowed). If you need the key, use `name` through iteration, or use `agg` / `transform`.
- **`observed=True` is the default** for categorical keys. Add `observed=False` to see empty categories.
- **`groupby(...).mean()` raises an error** when text columns are present. Select columns first or use `numeric_only=True`.
- Time frequency aliases: month end is **`"ME"`**, quarter end is **`"QE"`**, year end `"YE"` (the old `"M"`, `"Q"`, `"Y"` are no longer accepted).
- Group keys that are NaN are **dropped** unless `dropna=False`.
- `groupby` results are new objects. Always assign them: `result = df.groupby(...).agg(...)`.

---
# 15. Common mistakes

| Mistake | Fix |
|---|---|
| Expecting `df.groupby("a")` to show a table | It is lazy. Add an aggregation (`.sum()`, `.agg(...)`) |
| Group keys end up in the index and you cannot use them as columns | `as_index=False` or `reset_index()` |
| `groupby("a").mean()` with text columns raises an error | `[["col1", "col2"]]` or `numeric_only=True` |
| Rows with NaN keys vanish | `dropna=False`, or clean the keys first |
| Confusing `count` and `size` | `count` ignores NaN, `size` counts rows |
| MultiIndex columns from the dict style of `agg` | Use named aggregation |
| Using `lambda` for sum/mean/max | Use the built-in string name (much faster) |
| Using `apply` when `transform` or `agg` is enough | Prefer `agg` / `transform` / `filter` |
| Using `agg` when you need a value on every row | Use `transform` |
| `transform` returning a different length | It must return one value per row (or a scalar) |
| Group statistics (means, rates) computed on all data before splitting | Learn them from the train set only |
| Target encoding on the same rows it was computed from | Train-only statistics, smoothing, out-of-fold |
| Using `groupby` + `shift`/`rolling` on unsorted data | Sort by group and time first (chapter 09) |
| Old frequency codes `"M"` | `"ME"` |

---
# 16. Interview questions

**Q1. Explain split-apply-combine.**
`groupby` splits the rows into groups by key(s), applies a function (aggregation, transformation or filter) to each group, and combines the results into one object.

**Q2. `agg` vs `transform` vs `apply` vs `filter`?**
`agg` returns one row per group, `transform` returns a result with the same length as the input (a group value on every row), `filter` keeps or removes whole groups, `apply` is the general-purpose, slower version for anything else.

**Q3. `size` vs `count`?**
`size` counts rows per group, including missing values. `count` counts non-missing values of a specific column.

**Q4. How do you do several aggregations with clean column names?**
Named aggregation: `df.groupby("k").agg(total=("a", "sum"), avg=("b", "mean"))`.

**Q5. How do you get the top-N rows in each group?**
Sort and use `groupby(...).head(N)`, or `groupby(...)["col"].rank(method="first", ascending=False) <= N`, or `apply(lambda g: g.nlargest(N, "col"))`.

**Q6. How do you add each group's mean as a new column?**
`df["group_mean"] = df.groupby("g")["x"].transform("mean")`.

**Q7. What happens to NaN group keys?**
They are dropped by default. Use `dropna=False` to keep them as a group.

**Q8. How do you group by month?**
`df.groupby(df["date"].dt.to_period("M"))` or `df.groupby(pd.Grouper(key="date", freq="ME"))`, or `resample("ME")` on a datetime index.

**Q9. Why is `groupby().agg(lambda ...)` slow?**
The lambda is a Python function called once per group. Built-in aggregations run in compiled code, so prefer the string names.

**Q10. How do you convert transaction data into one row per customer for ML?**
`groupby("customer_id").agg(...)` with named aggregation (frequency, total and average amount, recency from the last date, number of distinct categories), using only data before the prediction date.

**Q11. What is target encoding and what is its main risk?**
Replacing a category with the mean of the target for that category. If the mean is computed on the same rows (or on the test data) the feature contains the answer, which is leakage. Compute it from training data only, use smoothing and out-of-fold encoding.

**Q12. `groupby` vs `pivot_table`?**
`pivot_table` is a groupby with two keys shown as a grid (one key as rows, one as columns). `df.groupby([a, b])[x].agg(f).unstack()` gives the same result.

---
# 17. Practice questions

Run this cell first to get the practice data.

In [55]:
orders_p = pd.DataFrame({
    "order_id": range(1, 15),
    "customer": ["Ravi", "Asha", "Ravi", "Neha", "Amit", "Ravi", "Asha", "Sara", "Neha", "Ravi", "Amit", "Asha", "Sara", "Ravi"],
    "region":   ["North", "South", "North", "East", "South", "North", "South", "East", "East", "North", "South", "South", "East", "North"],
    "product":  ["Pen", "Book", "Bag", "Pen", "Bag", "Book", "Pen", "Book", "Bag", "Pen", "Book", "Bag", "Pen", "Book"],
    "qty":      [10, 2, 1, 25, 3, 5, 8, 2, 1, 12, 4, 2, 15, 3],
    "price":    [10, 250, 800, 10, 800, 250, 10, 250, 800, 10, 250, 800, 10, 250],
    "discount": [0, 10, np.nan, 5, 0, np.nan, 0, 15, 10, np.nan, 0, 5, np.nan, 10],
    "date":     pd.to_datetime(["2026-01-05", "2026-01-12", "2026-01-20", "2026-02-03", "2026-02-10", "2026-02-18", "2026-02-25",
                                "2026-03-02", "2026-03-09", "2026-03-15", "2026-03-22", "2026-04-01", "2026-04-07", "2026-04-15"]),
})
orders_p["revenue"] = orders_p["qty"] * orders_p["price"] * (1 - orders_p["discount"].fillna(0) / 100)
orders_p

,order_id,customer,region,product,qty,price,discount,date,revenue
0,1,Ravi,North,Pen,10,10,0.0,2026-01-05,100.0
1,2,Asha,South,Book,2,250,10.0,2026-01-12,450.0
2,3,Ravi,North,Bag,1,800,NaN,2026-01-20,800.0
3,4,Neha,East,Pen,25,10,5.0,2026-02-03,237.5
4,5,Amit,South,Bag,3,800,0.0,2026-02-10,2400.0
5,6,Ravi,North,Book,5,250,NaN,2026-02-18,1250.0
6,7,Asha,South,Pen,8,10,0.0,2026-02-25,80.0
7,8,Sara,East,Book,2,250,15.0,2026-03-02,425.0
8,9,Neha,East,Bag,1,800,10.0,2026-03-09,720.0
9,10,Ravi,North,Pen,12,10,NaN,2026-03-15,120.0


**Q1.** Total revenue per region.
**Q2.** For each product: number of orders, total quantity and average revenue (named aggregation).
**Q3.** Revenue by region **and** product, as a normal table (no MultiIndex).
**Q4.** Which region has the highest total revenue? (`idxmax`)
**Q5.** Add `share_in_region` = each order's revenue as a fraction of its region's total.
**Q6.** Keep only the customers who placed at least 3 orders (`filter`).
**Q7.** For each customer: first and last order date and number of days between them.
**Q8.** The top 2 orders by revenue inside each region.
**Q9.** Monthly total revenue (use `Grouper` or `resample`).
**Q10.** For each region show `size` (rows) and `count` of the `discount` column. Why are they different?
**Q11.** Build a customer feature table with cutoff date `2026-04-01` using only earlier orders: `frequency`, `monetary`, `avg_order`, `recency_days`. Add the label `bought_after`: whether the customer ordered on or after the cutoff.
**Q12.** Find customers whose **average order value** is higher than the average order value of **all** orders.

### Solutions

In [56]:
# Q1
orders_p.groupby("region")["revenue"].sum()

region
East     1532.5
North    2945.0
South    5450.0
Name: revenue, dtype: float64

In [57]:
# Q2
orders_p.groupby("product").agg(
    n_orders=("order_id", "count"),
    total_qty=("qty", "sum"),
    avg_revenue=("revenue", "mean"),
).round(1)

,n_orders,total_qty,avg_revenue
product,,,
Bag,4,7,1360.0
Book,5,16,760.0
Pen,5,70,137.5


In [58]:
# Q3
orders_p.groupby(["region", "product"], as_index=False)["revenue"].sum()

,region,product,revenue
0,East,Bag,720.0
1,East,Book,425.0
2,East,Pen,387.5
3,North,Bag,800.0
4,North,Book,1925.0
5,North,Pen,220.0
6,South,Bag,3920.0
7,South,Book,1450.0
8,South,Pen,80.0


In [59]:
# Q4
by_region = orders_p.groupby("region")["revenue"].sum()
print(by_region.idxmax(), by_region.max())

South 5450.0


In [60]:
# Q5
q = orders_p.copy()
q["share_in_region"] = (q["revenue"] / q.groupby("region")["revenue"].transform("sum")).round(3)
q[["order_id", "region", "revenue", "share_in_region"]].head(6)

,order_id,region,revenue,share_in_region
0,1,North,100.0,0.034
1,2,South,450.0,0.083
2,3,North,800.0,0.272
3,4,East,237.5,0.155
4,5,South,2400.0,0.440
5,6,North,1250.0,0.424


In [61]:
# Q6
orders_p.groupby("customer").filter(lambda g: len(g) >= 3)[["customer", "order_id"]].sort_values("customer")

,customer,order_id
1,Asha,2
6,Asha,7
11,Asha,12
0,Ravi,1
2,Ravi,3
5,Ravi,6
9,Ravi,10
13,Ravi,14


In [62]:
# Q7
c = orders_p.groupby("customer").agg(first_order=("date", "min"), last_order=("date", "max"))
c["days_between"] = (c["last_order"] - c["first_order"]).dt.days
c

,first_order,last_order,days_between
customer,,,
Amit,2026-02-10,2026-03-22,40
Asha,2026-01-12,2026-04-01,79
Neha,2026-02-03,2026-03-09,34
Ravi,2026-01-05,2026-04-15,100
Sara,2026-03-02,2026-04-07,36


In [63]:
# Q8
(
    orders_p.sort_values("revenue", ascending=False)
            .groupby("region")
            .head(2)
            .sort_values(["region", "revenue"], ascending=[True, False])
            [["region", "order_id", "product", "revenue"]]
)

,region,order_id,product,revenue
8,East,9,Bag,720.0
7,East,8,Book,425.0
5,North,6,Book,1250.0
2,North,3,Bag,800.0
4,South,5,Bag,2400.0
11,South,12,Bag,1520.0


In [64]:
# Q9
orders_p.groupby(pd.Grouper(key="date", freq="ME"))["revenue"].sum()

date
2026-01-31    1350.0
2026-02-28    3967.5
2026-03-31    2265.0
2026-04-30    2345.0
Freq: ME, Name: revenue, dtype: float64

In [65]:
# Q10
orders_p.groupby("region").agg(rows=("order_id", "size"), discount_count=("discount", "count"))
# size counts every row, count skips the NaN discounts

,rows,discount_count
region,,
East,4,3
North,5,2
South,5,5


In [66]:
# Q11
cut = pd.Timestamp("2026-04-01")
past = orders_p[orders_p["date"] < cut]
later = orders_p[orders_p["date"] >= cut]

ft = past.groupby("customer").agg(
    frequency=("order_id", "count"),
    monetary=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    last_order=("date", "max"),
)
ft["recency_days"] = (cut - ft["last_order"]).dt.days
ft = ft.drop(columns="last_order")
ft["bought_after"] = ft.index.isin(later["customer"]).astype(int)
ft.round(1)

,frequency,monetary,avg_order,recency_days,bought_after
customer,,,,,
Amit,2,3400.0,1700.0,10,0
Asha,2,530.0,265.0,35,1
Neha,2,957.5,478.8,23,0
Ravi,4,2270.0,567.5,17,1
Sara,1,425.0,425.0,30,1


In [67]:
# Q12
overall_avg = orders_p["revenue"].mean()
avg_by_customer = orders_p.groupby("customer")["revenue"].mean()
print("overall average:", round(overall_avg, 1))
avg_by_customer[avg_by_customer > overall_avg].round(1)

overall average: 709.1


customer
Amit    1700.0
Name: revenue, dtype: float64

---
# 18. Cheat sheet

```python
# BASIC
df.groupby("g")["x"].sum()                      # one key, one column
df.groupby(["g1", "g2"])["x"].mean()            # two keys -> MultiIndex
df.groupby("g", as_index=False)["x"].sum()      # keys stay as columns
df.groupby("g", sort=False, dropna=False)       # keep order / keep NaN keys
df.groupby("g").mean(numeric_only=True)         # skip text columns

# AGG (named aggregation is the best style)
df.groupby("g").agg(
    n=("x", "count"),
    total=("x", "sum"),
    avg=("y", "mean"),
    top=("z", lambda s: s.mode().iloc[0]),
)
df.groupby("g")["x"].agg(["min", "max", "mean"])

# COUNTS
.size()       # rows (incl. NaN)       .count()   # non-missing       .nunique()   # distinct

# SAME SHAPE as the original (features)
df["g_mean"] = df.groupby("g")["x"].transform("mean")
df["share"]  = df["x"] / df.groupby("g")["x"].transform("sum")
df["rank"]   = df.groupby("g")["x"].rank(ascending=False, method="first")

# FILTER / PICK ROWS
df.groupby("g").filter(lambda t: len(t) >= 3)
df.sort_values("x", ascending=False).groupby("g").head(2)
df.groupby("g").nth(0)        df.groupby("g").cumcount()     df.groupby("g").ngroup()

# OTHER KEYS
df.groupby(pd.Grouper(key="date", freq="ME"))["x"].sum()
df.groupby(df["date"].dt.month)["x"].sum()
df.groupby(pd.cut(df["x"], bins), observed=True)["y"].mean()

# GRID
df.pivot_table(index="a", columns="b", values="x", aggfunc="sum", fill_value=0)
df.groupby(["a", "b"])["x"].sum().unstack()

# ML PATTERN: transactions -> one row per entity (only data BEFORE the cutoff)
feat = history.groupby("customer_id").agg(frequency=("id", "count"), monetary=("amount", "sum"),
                                          avg_order=("amount", "mean"), last=("date", "max"))
feat["recency_days"] = (cutoff - feat["last"]).dt.days
# Target encoding: learn from TRAIN only, map to test, fill unseen with the global mean
rate = train.groupby("k")["y"].mean();   test["k_rate"] = test["k"].map(rate).fillna(train["y"].mean())
```

**Next chapter:** `11_merge_join_concat`: combining tables. `merge` (inner, left, right, outer), join keys, `join`, `concat`, duplicate keys, and how to avoid losing or multiplying rows.